# Open-Weight Model Audit — Download → Inspect → Measure → Generate

Default classroom model: Qwen3-0.6B.

Change MODEL_ID to a larger open-weight checkpoint only after the resource estimate is complete. See docs/OPEN_WEIGHT_QWEN_H100_CASE_STUDY.md for the H100 path.

Learning objective: take an unfamiliar checkpoint and produce a reproducible model audit before training it.


In [ ]:
!pip -q install -U transformers accelerate
import torch, transformers, time
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30,2))


## 1. Choose the artifact

Record the exact repository/model ID and revision before downloading.


In [ ]:
MODEL_ID = 'Qwen/Qwen3-0.6B'
print('MODEL_ID =', MODEL_ID)


## 2. Resource prediction before loading

Raw weight estimate = parameter count × bytes per parameter.
BF16 is approximately 2 bytes/parameter; 4-bit raw storage is approximately 0.5 bytes/parameter before metadata and packing overhead.


In [ ]:
for params_b in [0.6, 7, 27]:
    print(f'{params_b:g}B BF16 raw weights ≈ {params_b*2:.1f} GB')
    print(f'{params_b:g}B 4-bit raw weights ≈ {params_b*0.5:.1f} GB')


## 3. Inspect configuration and tokenizer


In [ ]:
from transformers import AutoConfig, AutoTokenizer
config = AutoConfig.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
for name in ['model_type','vocab_size','hidden_size','num_hidden_layers','num_attention_heads','num_key_value_heads','max_position_embeddings']:
    print(name, getattr(config, name, None))


## 4. Load and measure memory

Compare the raw-weight estimate with actual runtime allocation.


In [ ]:
from transformers import AutoModelForCausalLM
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map='auto' if torch.cuda.is_available() else None)
if torch.cuda.is_available(): torch.cuda.synchronize()
print('parameter count:', sum(p.numel() for p in model.parameters()))
print('load seconds:', round(time.perf_counter()-t0,2))
if torch.cuda.is_available():
    print('allocated GiB:', round(torch.cuda.memory_allocated()/2**30,2))
    print('reserved GiB:', round(torch.cuda.memory_reserved()/2**30,2))


## 5. Baseline generation and throughput


In [ ]:
messages=[{'role':'user','content':'Explain what a training checkpoint contains in three sentences.'}]
inputs=tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_tensors='pt').to(model.device)
t0=time.perf_counter()
with torch.no_grad(): out=model.generate(**inputs, max_new_tokens=64, do_sample=False)
if torch.cuda.is_available(): torch.cuda.synchronize()
dt=time.perf_counter()-t0
generated=out.shape[-1]-inputs.shape[-1]
print(tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True))
print('prompt tokens:', inputs.shape[-1], 'generated tokens:', generated, 'seconds:', round(dt,3), 'output tok/s:', round(generated/max(dt,1e-9),2))


## 6. Break experiment

Choose one: longer prompt, longer generation, higher concurrency, or a larger model ID.

Predict the dominant resource term before running.

Required audit record:
model ID + revision → tokenizer → architecture → parameter count → precision → raw-weight estimate → measured memory → prompt/output tokens → latency → tokens/sec → training decision


## Next step

Move to notebooks/sft_with_a_small_open_model.ipynb only after this audit is complete.

Record the run with templates/EXPERIMENT_CARD.md.
